In [15]:
import json
import pandas as pd
import numpy as np
import re
from typing import Dict, List, Any, Optional, Union
from pathlib import Path


class FootballEventsProcessor:
    """
    Procesa archivos de eventos de fútbol JSONP y los convierte en DataFrames estructurados.
    Incluye cálculo de carries entre eventos consecutivos y Expected Threat (xT).

    Attributes:
        event_codes (dict): Mapeo de typeId a información del evento
        qualifier_codes (dict): Mapeo de qualifierId a información del qualifier
        qualifier_columns (dict): Columnas dinámicas para qualifiers
        player_jersey_mapping (dict): Mapeo de playerId a jersey number
    """

    # ------------------------------------------------------------------
    # xT grid: 8 rows (Y axis, 0-68) × 12 columns (X axis, 0-105).
    # Row 0 = bottom of pitch (y≈0), row 7 = top (y≈68).
    # Column 0 = own goal line (x=0), column 11 = opponent goal line (x=105).
    # ------------------------------------------------------------------
    XT_GRID = np.array([
        [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.0174506,  0.02122129, 0.02756312, 0.03485072, 0.0379259 ],
        [0.00750072, 0.00878589, 0.00942382, 0.0105949,  0.01214719, 0.0138454,  0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
        [0.0088799,  0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.0241224,  0.02855202, 0.05491138, 0.06442595],
        [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.0199707,  0.02385149, 0.03511326, 0.10805102, 0.25745362],
        [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.0199707,  0.02385149, 0.03511326, 0.10805102, 0.25745362],
        [0.0088799,  0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.0241224,  0.02855202, 0.05491138, 0.06442595],
        [0.00750072, 0.00878589, 0.00942382, 0.0105949,  0.01214719, 0.0138454,  0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
        [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.0174506,  0.02122129, 0.02756312, 0.03485072, 0.0379259 ],
    ])

    def __init__(self, events_mapping_path: str, qualifiers_mapping_path: str):
        """
        Inicializa el procesador con los archivos de mapeo.

        Args:
            events_mapping_path: Ruta al archivo de mapeo de eventos (OptaEvents)
            qualifiers_mapping_path: Ruta al archivo de mapeo de qualifiers
        """
        self.event_codes = self._load_mapping_file(events_mapping_path)
        self.qualifier_codes = self._load_mapping_file(qualifiers_mapping_path)

        self.qualifier_columns = self._create_qualifier_columns()
        self.player_jersey_mapping = {}

        # {teamId: {'teamName': str, 'oppositionTeamName': str, 'h_a': str}}
        self.team_mapping: Dict[str, Dict[str, str]] = {}

        self.basic_columns_mapping = {
            'eventId': 'eventId',
            'timeMin': 'minute',
            'timeSec': 'second',
            'contestantId': 'teamId',
            'playerId': 'playerId',
            'playerName': 'playerName',
            'x': 'x',
            'y': 'y',
            'periodId': 'period',
            'id': 'id',
            'typeId': 'typeId',
            'timeStamp': 'timeStamp',
            'outcome': 'outcomeType'
        }

    # -------------------------------------------------------------------------
    # MAPPING / LOADING
    # -------------------------------------------------------------------------

    def _load_mapping_file(self, file_path: str) -> Dict[int, Dict[str, str]]:
        """
        Carga archivo de mapeo JavaScript y extrae el diccionario.

        Args:
            file_path: Ruta al archivo de mapeo

        Returns:
            Diccionario con el mapeo {id: {name, description, value}}
        """
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()

            pattern = r'var\s+\w+\s*=\s*(\{.*\});?'
            match = re.search(pattern, content, re.DOTALL)

            if match:
                js_object = match.group(1)
                js_object = re.sub(r'(\w+):', r'"\1":', js_object)
                js_object = re.sub(r"'([^']*)'", r'"\1"', js_object)

                mapping_dict = json.loads(js_object)
                return {int(k): v for k, v in mapping_dict.items()}
            else:
                raise ValueError(f"No se pudo extraer el mapeo de {file_path}")

        except Exception as e:
            print(f"❌ Error cargando mapeo desde {file_path}: {e}")
            return {}

    def _create_qualifier_columns(self) -> Dict[str, List[str]]:
        """
        Crea las columnas dinámicas para todos los qualifiers posibles.

        Returns:
            Diccionario con información de las columnas de qualifiers
        """
        type_value_columns = []
        value_columns = []

        for qualifier_id, qualifier_info in self.qualifier_codes.items():
            qualifier_name = qualifier_info['name']
            type_value_columns.append(f"type_value_{qualifier_name}")
            value_columns.append(f"value_{qualifier_name}")

        return {
            'type_value': type_value_columns,
            'value': value_columns,
            'all': type_value_columns + value_columns
        }

    # -------------------------------------------------------------------------
    # JSON / JSONP PARSING
    # -------------------------------------------------------------------------

    def _extract_json_from_jsonp(self, file_path: str) -> Dict[str, Any]:
        """
        Extrae JSON de archivo JSONP eliminando el wrapper de función.

        Args:
            file_path: Ruta al archivo JSONP

        Returns:
            Diccionario con los datos JSON
        """
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        pattern = r'^[^(]+\((.*)\)$'
        match = re.search(pattern, content, re.DOTALL)

        if match:
            json_str = match.group(1)
            return json.loads(json_str)
        else:
            raise ValueError("No se pudo extraer JSON del archivo JSONP")

    # -------------------------------------------------------------------------
    # JERSEY NUMBER MAPPING
    # -------------------------------------------------------------------------

    def _extract_player_jersey_mapping(self, events: List[Dict[str, Any]]) -> Dict[str, int]:
        """
        Extrae el mapeo de playerId a jersey number desde los eventos de Team Set Up (typeId: 34).

        Args:
            events: Lista de todos los eventos del archivo

        Returns:
            Diccionario con el mapeo {playerId: jerseyNumber}
        """
        player_jersey_map = {}

        setup_events = [event for event in events if event.get('typeId') == 34]

        if not setup_events:
            print("Advertencia: No se encontraron eventos de Team Set Up (typeId: 34)")
            return player_jersey_map

        for setup_event in setup_events:
            team_id = setup_event.get('contestantId')
            qualifiers = setup_event.get('qualifier', [])

            player_ids_str = None
            jersey_numbers_str = None

            for qualifier in qualifiers:
                qualifier_id = qualifier.get('qualifierId')
                if qualifier_id == 30:
                    player_ids_str = qualifier.get('value', '')
                elif qualifier_id == 59:
                    jersey_numbers_str = qualifier.get('value', '')

            if player_ids_str and jersey_numbers_str:
                player_ids = [pid.strip() for pid in player_ids_str.split(',')]
                jersey_numbers = [jn.strip() for jn in jersey_numbers_str.split(',')]

                if len(player_ids) != len(jersey_numbers):
                    print(f"Advertencia: Desajuste en cantidad de IDs ({len(player_ids)}) "
                          f"y números de camiseta ({len(jersey_numbers)}) para team {team_id}")
                    continue

                for player_id, jersey_num in zip(player_ids, jersey_numbers):
                    if player_id and jersey_num:
                        try:
                            player_jersey_map[player_id] = int(jersey_num)
                        except ValueError:
                            player_jersey_map[player_id] = 0

        print(f"Mapeo de jersey numbers creado: {len(player_jersey_map)} jugadores")
        print(player_jersey_map)
        return player_jersey_map

    # -------------------------------------------------------------------------
    # TEAM MAPPING
    # -------------------------------------------------------------------------

    def _extract_team_mapping(self, contestants: List[Dict[str, Any]]) -> Dict[str, Dict[str, str]]:
        """
        Construye el mapeo de equipos a partir de la lista 'contestant' del
        bloque matchInfo del archivo JSONP.

        Para cada equipo se almacena:
          - teamName          : nombre del club (campo 'name')
          - oppositionTeamName: nombre del rival
          - h_a               : 'home' o 'away' (campo 'position')

        Args:
            contestants: Lista de dicts con la información de cada equipo.

        Returns:
            Diccionario {teamId: {'teamName': str, 'oppositionTeamName': str, 'h_a': str}}
        """
        if not contestants or len(contestants) != 2:
            print(f"⚠️ Advertencia: Se esperaban exactamente 2 equipos, "
                  f"se encontraron {len(contestants) if contestants else 0}")
            return {}

        team_mapping = {}

        for contestant in contestants:
            team_id   = contestant.get('id')
            team_name = contestant.get('name')
            h_a       = contestant.get('position')   # 'home' | 'away'

            if not team_id or not team_name:
                print(f"⚠️ Advertencia: Datos de equipo incompletos: {contestant}")
                continue

            team_mapping[team_id] = {
                'teamName': team_name,
                'h_a':      h_a,
                'oppositionTeamName': None  # Se rellena en el segundo paso
            }

        # Rellenar oppositionTeamName cruzando los dos equipos
        if len(team_mapping) == 2:
            ids = list(team_mapping.keys())
            team_mapping[ids[0]]['oppositionTeamName'] = team_mapping[ids[1]]['teamName']
            team_mapping[ids[1]]['oppositionTeamName'] = team_mapping[ids[0]]['teamName']

        print(f"✅ Mapeo de equipos creado: "
              + " vs ".join(f"{v['teamName']} ({v['h_a']})" for v in team_mapping.values()))

        return team_mapping



    def _extract_basic_columns(self, event: Dict[str, Any]) -> Dict[str, Any]:
        """
        Extrae las columnas básicas de un evento.

        Args:
            event: Diccionario con los datos del evento

        Returns:
            Diccionario con las columnas básicas mapeadas
        """
        row = {}

        for json_key, df_column in self.basic_columns_mapping.items():
            row[df_column] = event.get(json_key, None)

        type_id = event.get('typeId')
        if type_id is not None:
            event_info = self.get_event_info(type_id)
            row['type'] = event_info['name']
        else:
            row['type'] = None

        player_id = row.get('playerId')
        if player_id and player_id in self.player_jersey_mapping:
            row['player_JerseyNumber'] = self.player_jersey_mapping[player_id]
        elif player_id:
            print(f"⚠️ Advertencia: playerId '{player_id}' no encontrado en el mapeo de jersey numbers")
            row['player_JerseyNumber'] = None
        else:
            row['player_JerseyNumber'] = None

        # Team columns: teamName, oppositionTeamName, h_a
        team_id = row.get('teamId')
        if team_id and team_id in self.team_mapping:
            team_info = self.team_mapping[team_id]
            row['teamName']           = team_info['teamName']
            row['oppositionTeamName'] = team_info['oppositionTeamName']
            row['h_a']                = team_info['h_a']
        else:
            row['teamName']           = None
            row['oppositionTeamName'] = None
            row['h_a']                = None

        for col_name in self.qualifier_columns['all']:
            row[col_name] = None

        self._process_event_qualifiers(event, row)
        self._create_special_qualifier_columns(row)
        self._apply_fallback_coordinates(row)

        return row

    def _process_event_qualifiers(self, event: Dict[str, Any], row: Dict[str, Any]) -> None:
        """
        Procesa los qualifiers de un evento específico y llena las columnas correspondientes.

        Args:
            event: Diccionario con los datos del evento
            row: Fila del DataFrame a llenar
        """
        qualifiers = event.get('qualifier', [])

        for qualifier in qualifiers:
            qualifier_id = qualifier.get('qualifierId')
            qualifier_value = qualifier.get('value')

            if qualifier_id is not None:
                qualifier_info = self.get_qualifier_info(qualifier_id)
                qualifier_name = qualifier_info['name']

                type_value_col = f"type_value_{qualifier_name}"
                value_col = f"value_{qualifier_name}"

                if type_value_col in row:
                    row[type_value_col] = qualifier_id
                if value_col in row:
                    row[value_col] = qualifier_value

    def _create_special_qualifier_columns(self, row: Dict[str, Any]) -> None:
        """
        Crea columnas especiales basadas en valores específicos de qualifiers.
        Convierte coordenadas de 0-100 a dimensiones reales del campo (105x68).

        Args:
            row: Fila del DataFrame a modificar
        """
        special_columns_mapping = {
            'endX': 'value_Pass End X',
            'endY': 'value_Pass End Y',
            'blockedX': 'value_Blocked x co-ordinate',
            'blockedY': 'value_Blocked y co-ordinate',
            'goalMouthZ': 'value_Goal mouth z coordinate',
            'goalMouthY': 'value_Goal mouth y coordinate'
        }

        x_coordinate_columns = ['endX', 'blockedX', 'x']
        y_coordinate_columns = ['endY', 'blockedY', 'y']

        if 'x' in row and row['x'] is not None:
            try:
                row['x'] = (float(row['x']) / 100) * 105
            except (ValueError, TypeError):
                row['x'] = None

        if 'y' in row and row['y'] is not None:
            try:
                row['y'] = (float(row['y']) / 100) * 68
            except (ValueError, TypeError):
                row['y'] = None

        for special_col, qualifier_col in special_columns_mapping.items():
            row[special_col] = None
            if qualifier_col in row and row[qualifier_col] is not None:
                try:
                    value = float(row[qualifier_col])
                    if special_col in x_coordinate_columns:
                        row[special_col] = (value / 100) * 105
                    elif special_col in y_coordinate_columns:
                        row[special_col] = (value / 100) * 68
                    else:
                        row[special_col] = value
                except (ValueError, TypeError):
                    row[special_col] = None

    def _apply_fallback_coordinates(self, row: Dict[str, Any]) -> None:
        """
        Usa x/y como respaldo para endX/endY cuando estos son None.

        Args:
            row: Fila del DataFrame a modificar
        """
        if row['endX'] is None and row['x'] is not None:
            row['endX'] = float(row['x'])
        if row['endY'] is None and row['y'] is not None:
            row['endY'] = float(row['y'])

    # -------------------------------------------------------------------------
    # CARRY CALCULATION
    # -------------------------------------------------------------------------

    def _create_carry_row(
        self,
        event1: pd.Series,
        event2: pd.Series,
        start_x: float,
        start_y: float,
        end_x: float,
        end_y: float,
        player_id: Any,
        player_name: Optional[str],
        player_jersey: Optional[int],
        team_id: Any,
        period: Any
    ) -> Dict[str, Any]:
        """
        Crea una fila de evento carry con valores interpolados.

        Args:
            event1: Evento previo al carry
            event2: Evento posterior al carry
            start_x/y: Coordenadas de inicio del carry
            end_x/y: Coordenadas de fin del carry
            player_id: ID del jugador que realiza el carry
            player_name: Nombre del jugador
            player_jersey: Número de camiseta
            team_id: ID del equipo
            period: Período del partido

        Returns:
            Diccionario representando el evento carry
        """
        avg_minute = (event1['minute'] + event2['minute']) / 2
        avg_second = (event1['second'] + event2['second']) / 2

        if avg_second >= 60:
            avg_minute += 1
            avg_second -= 60

        # Resolve team info from the mapping (already populated at this stage)
        team_info = self.team_mapping.get(team_id, {})

        return {
            'eventId': np.random.randint(100000, 999999),
            'minute': int(avg_minute),
            'second': avg_second,
            'teamId': team_id,
            'teamName': team_info.get('teamName'),
            'oppositionTeamName': team_info.get('oppositionTeamName'),
            'h_a': team_info.get('h_a'),
            'x': start_x,
            'y': start_y,
            'endX': end_x,
            'endY': end_y,
            'period': period,
            'type': 'Carry',
            'playerId': player_id,
            'playerName': player_name,
            'player_JerseyNumber': player_jersey,
            'outcomeType': 1
        }

    def calculate_carries(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calcula e inserta eventos carry entre acciones consecutivas.

        Args:
            df: DataFrame de eventos ordenado por minuto y segundo

        Returns:
            DataFrame original con los eventos carry insertados
        """
        carries_to_insert = []

        for i in range(len(df) - 1):
            current_event = df.iloc[i]
            next_event = df.iloc[i + 1]

            current_type = current_event['type']
            next_type = next_event['type']
            current_outcome = current_event.get('outcomeType', None)
            next_outcome = next_event.get('outcomeType', None)

            same_team = current_event['teamId'] == next_event['teamId']

            # Check coordinate mismatch
            coords_dont_match = False
            if pd.notna(current_event.get('endX')) and pd.notna(current_event.get('endY')):
                if pd.notna(next_event.get('x')) and pd.notna(next_event.get('y')):
                    coords_dont_match = (
                        abs(current_event['endX'] - next_event['x']) > 0.01 or
                        abs(current_event['endY'] - next_event['y']) > 0.01
                    )

            create_carry = False
            carry_start_x = current_event.get('endX')
            carry_start_y = current_event.get('endY')
            carry_end_x = next_event.get('x')
            carry_end_y = next_event.get('y')

            # --- Edge cases: never create carry ---
            if next_type in ['BallTouch', 'BallRecovery', 'Aerial', 'CornerAwarded']:
                continue
            if current_type in ['Foul', 'Card']:
                continue
            if current_type == 'MissedShot' and next_type == 'BallTouch':
                continue

            # --- Tackle ---
            if current_type == 'Tackle':
                if next_type == 'BallRecovery' and current_event.get('playerId') != next_event.get('playerId'):
                    continue
                elif next_type == 'Pass' and current_event.get('playerId') == next_event.get('playerId') and same_team and coords_dont_match:
                    create_carry = True

            # --- Pass ---
            if current_type == 'Pass' and same_team and coords_dont_match and current_outcome == 1:
                if next_type == 'Pass':
                    create_carry = True
                elif next_type in ['Shot', 'MissedShot', 'SavedShot', 'Dispossessed', 'Foul']:
                    create_carry = True

            # --- Ball recoveries / keeper / interceptions ---
            if current_type in ['BallRecovery', 'KeeperPickup', 'Interception', 'Claim']:
                if same_team and coords_dont_match and next_type in ['Pass', 'Shot', 'MissedShot', 'SavedShot']:
                    create_carry = True

            # --- Clearance ---
            if current_type == 'Clearance' and next_type == 'Pass' and same_team and coords_dont_match:
                create_carry = True

            # --- Before Dispossessed / Foul / SavedShot ---
            if next_type in ['Dispossessed', 'Foul', 'SavedShot']:
                if current_type == 'Pass' and same_team and coords_dont_match and current_outcome == 1:
                    create_carry = True

            # --- Special: Challenge unsuccessful → TakeOn successful (different teams) ---
            if current_type == 'Challenge' and current_outcome == 0:
                if next_type == 'Take On' and next_outcome == 1 and current_event['teamId'] != next_event['teamId']:
                    if i > 0:
                        prev_event = df.iloc[i - 1]
                        if (prev_event['teamId'] == next_event['teamId'] and
                                pd.notna(prev_event.get('endX')) and pd.notna(prev_event.get('endY')) and
                                pd.notna(next_event.get('x')) and pd.notna(next_event.get('y'))):
                            if (abs(prev_event['endX'] - next_event['x']) > 0.01 or
                                    abs(prev_event['endY'] - next_event['y']) > 0.01):
                                carries_to_insert.append((i + 1, self._create_carry_row(
                                    prev_event, next_event,
                                    prev_event['endX'], prev_event['endY'],
                                    next_event['x'], next_event['y'],
                                    next_event['playerId'], next_event.get('playerName'),
                                    next_event.get('player_JerseyNumber'),
                                    next_event['teamId'], next_event['period']
                                )))

                    if i + 2 < len(df):
                        next_next_event = df.iloc[i + 2]
                        if (next_next_event['teamId'] == next_event['teamId'] and
                                pd.notna(next_event.get('endX')) and pd.notna(next_event.get('endY')) and
                                pd.notna(next_next_event.get('x')) and pd.notna(next_next_event.get('y'))):
                            if (abs(next_event['endX'] - next_next_event['x']) > 0.01 or
                                    abs(next_event['endY'] - next_next_event['y']) > 0.01):
                                carries_to_insert.append((i + 2, self._create_carry_row(
                                    next_event, next_next_event,
                                    next_event['endX'], next_event['endY'],
                                    next_next_event['x'], next_next_event['y'],
                                    next_event['playerId'], next_event.get('playerName'),
                                    next_event.get('player_JerseyNumber'),
                                    next_event['teamId'], next_event['period']
                                )))
                    continue

            # --- Special: TakeOn successful → Challenge unsuccessful ---
            if current_type == 'Take On' and current_outcome == 1:
                if next_type == 'Challenge' and next_outcome == 0 and current_event['teamId'] != next_event['teamId']:
                    if i > 0:
                        prev_event = df.iloc[i - 1]
                        if (prev_event['teamId'] == current_event['teamId'] and
                                pd.notna(prev_event.get('endX')) and pd.notna(prev_event.get('endY')) and
                                pd.notna(current_event.get('x')) and pd.notna(current_event.get('y'))):
                            if (abs(prev_event['endX'] - current_event['x']) > 0.01 or
                                    abs(prev_event['endY'] - current_event['y']) > 0.01):
                                carries_to_insert.append((i, self._create_carry_row(
                                    prev_event, current_event,
                                    prev_event['endX'], prev_event['endY'],
                                    current_event['x'], current_event['y'],
                                    current_event['playerId'], current_event.get('playerName'),
                                    current_event.get('player_JerseyNumber'),
                                    current_event['teamId'], current_event['period']
                                )))

                    if i + 2 < len(df):
                        next_next_event = df.iloc[i + 2]
                        if (next_next_event['teamId'] == current_event['teamId'] and
                                pd.notna(current_event.get('endX')) and pd.notna(current_event.get('endY')) and
                                pd.notna(next_next_event.get('x')) and pd.notna(next_next_event.get('y'))):
                            if (abs(current_event['endX'] - next_next_event['x']) > 0.01 or
                                    abs(current_event['endY'] - next_next_event['y']) > 0.01):
                                carries_to_insert.append((i + 2, self._create_carry_row(
                                    current_event, next_next_event,
                                    current_event['endX'], current_event['endY'],
                                    next_next_event['x'], next_next_event['y'],
                                    current_event['playerId'], current_event.get('playerName'),
                                    current_event.get('player_JerseyNumber'),
                                    current_event['teamId'], current_event['period']
                                )))
                    continue

            # --- Standard carry insertion ---
            if create_carry and all(pd.notna(v) for v in [carry_start_x, carry_start_y, carry_end_x, carry_end_y]):
                carries_to_insert.append((i + 1, self._create_carry_row(
                    current_event, next_event,
                    carry_start_x, carry_start_y,
                    carry_end_x, carry_end_y,
                    next_event['playerId'], next_event.get('playerName'),
                    next_event.get('player_JerseyNumber'),
                    next_event['teamId'], next_event['period']
                )))

        # Insert carries in reverse order to preserve indices
        df_with_carries = df.copy()
        for idx, carry_row in sorted(carries_to_insert, key=lambda x: x[0], reverse=True):
            df_with_carries = pd.concat([
                df_with_carries.iloc[:idx],
                pd.DataFrame([carry_row]),
                df_with_carries.iloc[idx:]
            ]).reset_index(drop=True)

        print(f"✅ {len(carries_to_insert)} carries insertados en el DataFrame")
        return df_with_carries

    # -------------------------------------------------------------------------
    # EXPECTED THREAT (xT)
    # -------------------------------------------------------------------------

    def _get_xt_value(self, x: Optional[float], y: Optional[float]) -> Optional[float]:
        """
        Devuelve el valor xT de la celda del grid que corresponde a las
        coordenadas reales del campo (X: 0-105, Y: 0-68).

        La lógica de indexación es:
          col = floor(x / 105 * 12)  →  [0, 11]
          row = floor(y / 68  *  8)  →  [0,  7]

        Args:
            x: Coordenada X en el campo real (0-105)
            y: Coordenada Y en el campo real (0-68)

        Returns:
            Valor xT de esa celda, o None si las coordenadas son inválidas.
        """
        if x is None or y is None or np.isnan(x) or np.isnan(y):
            return None

        col = int(np.clip(x / 105 * 12, 0, 11))
        row = int(np.clip(y / 68  *  8, 0,  7))

        return float(self.XT_GRID[row, col])

    def calculate_xt(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calcula Expected Threat (xT) para eventos de tipo 'Pass' y 'Carry'.

        Añade tres columnas al DataFrame completo (NaN para el resto de tipos):
          - start_zone_value_xT : valor xT de la celda de inicio  (x, y)
          - end_zone_value_xT   : valor xT de la celda de fin     (endX, endY)
          - xT                  : diferencia end - start

        Args:
            df: DataFrame con todos los eventos (ya con carries si procede)

        Returns:
            DataFrame con las tres nuevas columnas añadidas.
        """
        df = df.copy()

        # Inicializar columnas como NaN
        df['start_zone_value_xT'] = np.nan
        df['end_zone_value_xT']   = np.nan
        df['xT']                  = np.nan

        mask = df['type'].isin(['Pass', 'Carry'])

        if not mask.any():
            print("⚠️ No se encontraron eventos de tipo 'Pass' o 'Carry' para calcular xT")
            return df

        df.loc[mask, 'start_zone_value_xT'] = df.loc[mask].apply(
            lambda row: self._get_xt_value(row['x'], row['y']), axis=1
        )
        df.loc[mask, 'end_zone_value_xT'] = df.loc[mask].apply(
            lambda row: self._get_xt_value(row['endX'], row['endY']), axis=1
        )
        df.loc[mask, 'xT'] = (
            df.loc[mask, 'end_zone_value_xT'] - df.loc[mask, 'start_zone_value_xT']
        )

        xt_events   = mask.sum()
        xt_computed = df.loc[mask, 'xT'].notna().sum()
        print(f"✅ xT calculado: {xt_computed}/{xt_events} eventos Pass/Carry con coordenadas válidas")

        return df

    # -------------------------------------------------------------------------
    # MAIN PROCESSING PIPELINE
    # -------------------------------------------------------------------------

    def process_events_file(
        self,
        file_path: str,
        include_carries: bool = True,
        include_xt: bool = True
    ) -> pd.DataFrame:
        """
        Procesa un archivo de eventos y devuelve un DataFrame estructurado.
        Opcionalmente calcula carries y Expected Threat (xT).

        El orden del pipeline es:
          1. Parsear JSONP y extraer datos
          2. Mapear equipos (teamName, oppositionTeamName, h_a)
          3. Mapear jersey numbers
          4. Construir columnas básicas + qualifiers + coordenadas
          5. (Opcional) Calcular e insertar carries
          6. (Opcional) Calcular xT para Pass y Carry

        Args:
            file_path     : Ruta al archivo JSONP de eventos
            include_carries: Si True, calcula e inserta carries (default: True)
            include_xt    : Si True, calcula columnas xT para Pass/Carry (default: True).
                            Requiere include_carries=True para que los carries
                            también reciban su valor xT.

        Returns:
            DataFrame con los eventos procesados
        """
        try:
            data = self._extract_json_from_jsonp(file_path)

            if 'liveData' not in data or 'event' not in data['liveData']:
                raise ValueError("No se encontraron eventos en el archivo")

            events = data['liveData']['event']

            # Extract team mapping from matchInfo (must come before event processing)
            contestants = data.get('matchInfo', {}).get('contestant', [])
            self.team_mapping = self._extract_team_mapping(contestants)

            self.player_jersey_mapping = self._extract_player_jersey_mapping(events)

            processed_events = []
            skipped_unknown = 0

            for event in events:
                row = self._extract_basic_columns(event)
                if row.get('type') == 'Unknown':
                    skipped_unknown += 1
                    continue
                processed_events.append(row)

            if skipped_unknown > 0:
                print(f"⚠️ Se omitieron {skipped_unknown} eventos con tipo 'Unknown'")

            df = pd.DataFrame(processed_events)

            print(f"✅ Procesados {len(df)} eventos exitosamente")
            print(f"Total de columnas creadas: {len(df.columns)}")

            basic_cols = [c for c in df.columns if not c.startswith(('type_value_', 'value_')) and c not in ['endX', 'endY', 'blockedX', 'blockedY', 'goalMouthZ', 'goalMouthY']]
            qualifier_cols = [c for c in df.columns if c.startswith(('type_value_', 'value_'))]
            special_cols = [c for c in df.columns if c in ['endX', 'endY', 'blockedX', 'blockedY', 'goalMouthZ', 'goalMouthY']]

            print(f"   - Columnas básicas: {len(basic_cols)} (incluye player_JerseyNumber)")
            print(f"   - Columnas qualifiers: {len(qualifier_cols)}")
            print(f"   - Columnas especiales: {len(special_cols)} {special_cols}")
            print(f"     (Conversión de coordenadas: X: 0-100 → 0-105, Y: 0-100 → 0-68)")

            if include_carries:
                df = self.calculate_carries(df)

            if include_xt:
                df = self.calculate_xt(df)

            return df

        except Exception as e:
            print(f"❌ Error procesando archivo {file_path}: {e}")
            return pd.DataFrame()

    # -------------------------------------------------------------------------
    # LOOKUP HELPERS
    # -------------------------------------------------------------------------

    def get_event_info(self, type_id: int) -> Dict[str, str]:
        """
        Obtiene información de un evento por su typeId.

        Args:
            type_id: ID del tipo de evento

        Returns:
            Diccionario con name y description del evento
        """
        return self.event_codes.get(type_id, {'name': 'Unknown', 'description': 'Unknown event type'})

    def get_qualifier_info(self, qualifier_id: int) -> Dict[str, str]:
        """
        Obtiene información de un qualifier por su qualifierId.

        Args:
            qualifier_id: ID del qualifier

        Returns:
            Diccionario con name, value y description del qualifier
        """
        return self.qualifier_codes.get(qualifier_id, {'name': 'Unknown', 'value': '', 'description': 'Unknown qualifier'})

In [16]:
# Inicializar procesador
processor = FootballEventsProcessor(
        events_mapping_path=r"C:\Users\Usuario\OneDrive\football-analytics-research\data\mapping\opta-events.js",
        qualifiers_mapping_path=r"C:\Users\Usuario\OneDrive\football-analytics-research\data\mapping\opta-qualifiers.js"
    )
    
    # Procesar archivo de eventos
df = processor.process_events_file(r"C:\Users\Usuario\OneDrive\football-analytics-research\data\raw\alaves_espanyol")

    # Mostrar resultados
print("\nColumnas del DataFrame:")
print(df.columns.tolist())
    
print("\nPrimeros eventos:")
print(df.head())
    
print(f"\nTotal de eventos: {len(df)}")
    
    # Ejemplo de uso de mapeos
if not df.empty:
    sample_type_id = df['typeId'].iloc[0]
    event_info = processor.get_event_info(sample_type_id)
    print(f"\nEjemplo de mapeo - Evento {sample_type_id}: {event_info['name']}")

✅ Mapeo de equipos creado: Deportivo Alavés (home) vs Espanyol (away)
Mapeo de jersey numbers creado: 45 jugadores
{'5f8kjgsaabn2li5r7bvpebbsa': 1, '7h0b81yjiiu356rycsakgd0yc': 23, '5djkwcoflvxcv5oida6txfyp6': 22, '7y8ns61qioqu7kfsqzkvkwhuy': 19, 'cddggachzf198p1yhq9owtofd': 4, '6810qbng9pfg1bztczv11grmd': 6, '59u1gn30f6bg5lr06zsglykq2': 17, '74wjzeja92l4q8fpy1bkmcsfd': 10, '5fqvl0m1rkzhj9tmafrcxur6c': 2, '9taxidttdt94nmvc3mqhqq0ut': 20, 'e5qp6dh1prc55llew51zpy1gp': 7, 'eyhvr45vc1n690dxobcthmndh': 3, '2rqxh5mex336ci45jaw428y5l': 5, '8qvqdjw1d58mw7420lpmk7bah': 8, '1o4lfwkyx28j63ara4ye4wjdg': 9, '1ore4ght7ls393tcpbmah9dcl': 11, '6peka1v0xt2ctv0qtoaw9cslx': 12, '3nchjse4ej0qhilo395td7u1h': 14, 'c23q56z612clw2oxpordhrwiy': 16, '1whtyw1adggpz4j1xp7sjbeg9': 18, '1am59svvwz5idpdjq307dd5hw': 31, '1wylybleussatoxg8u6reao2y': 33, '3qdlqw5d6ra7th109n0g3rnfu': 38, 'c7i0lduij4vsnluxisdnaxsk5': 1, '9tw6rz3lfmn06fzsabtx9ei95': 14, 'c44zop1hvob3cmgu2grj8m0yy': 22, '4zve4017u2i3zvxlnjruv25t5': 8, 'br4

In [21]:
df.to_csv(r'C:\Users\Usuario\OneDrive\football-analytics-research\data\processed\eventing-alaves_espanyol.csv')